In [4]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [5]:
# ==========================================
# 1. ENTRADA DE DATOS (INPUT)
# ==========================================
#Creamos un Input para ingresar la cantidad de docenas elaboradas en el mes que se quiere analizar los costos.
try:
    Docenas_Elaboradas = float(input("INGRESE POR FAVOR EL TOTAL DE DOCENAS ELABORADAS ESTE MES : "))
except ValueError:
    print("Error: Por favor ingrese un número válido.")
    docenas_elaboradas = 0

INGRESE POR FAVOR EL TOTAL DE DOCENAS ELABORADAS ESTE MES : 50000


##COSTOS TOTAL Y UNITARIO DE MATERIA PRIMA

In [6]:
# ==========================================
# 2. PROCESAMIENTO: MATERIA PRIMA
# ==========================================
# Cargar archivo de Materia Prima
df_Materia_Prima = pd.read_excel('MATERIA_PRIMA.xlsx')

In [7]:
# Limpieza de datos
df_Materia_Prima.columns = df_Materia_Prima.columns.str.strip() #Limpiar espacios en los nombres de las columnas
df_Materia_Prima = df_Materia_Prima.dropna(how='all', axis=1) #Eliminar columnas que queden completamente vacías.
df_Materia_Prima = df_Materia_Prima.dropna(subset=['Materia Prima']) # Limpiar filas vacías que puedan quedar al final.

In [8]:
df_Materia_Prima.head(10)

,Materia Prima,Unidad,Precio por Unidad,Requerimiento por Docena
0,Harina,Kg,1500,0.490
1,Leche,Litro,2000,0.230
2,Manteca,kg,5000,0.200
3,Azucar,kg,1000,0.050
4,Huevo,Unidad,150,3.000
5,Levadura,Kg,1500,0.015
6,"Sal, Escencias, Etc.",Kg,1000,0.015


In [9]:
print(df_Materia_Prima.to_string(index=False))

       Materia Prima Unidad  Precio por Unidad  Requerimiento por Docena
              Harina     Kg               1500                     0.490
               Leche  Litro               2000                     0.230
             Manteca     kg               5000                     0.200
             Azucar      kg               1000                     0.050
              Huevo  Unidad                150                     3.000
            Levadura     Kg               1500                     0.015
Sal, Escencias, Etc.     Kg               1000                     0.015


In [10]:
# Cálculos
#Multiplicamos el precio por unidad de cada materia prima, por la cantidad que se requiere para elaborar una docena para obtener el costo por docena
# precio x cantidad.
df_Materia_Prima['Costo por docena'] = df_Materia_Prima['Precio por Unidad'] * df_Materia_Prima['Requerimiento por Docena']
Costo_Unitario_Total = df_Materia_Prima['Costo por docena'].sum()
Costo_Materia_Prima_Total = Docenas_Elaboradas * Costo_Unitario_Total


In [11]:
# Agregar filas de totales
#Creamos Dos filas nuevas para agregar a la Planilla el Costo Total y Unitario de la Materia Prima.
fila_unitario = {
    'Materia Prima': 'Costo Materia Prima Unitario',
    'Costo por docena': Costo_Unitario_Total
}

fila_total_mes = {
    'Materia Prima': 'Costo Materia Prima Total',
    'Costo por docena': Costo_Materia_Prima_Total
}
df_Materia_Prima = pd.concat([df_Materia_Prima, pd.DataFrame([fila_unitario, fila_total_mes])], ignore_index=True) #Realizamos la concatenación.

In [12]:
df_Materia_Prima.head(10) #Corroboramos que se realizaron todos los pasos correctamente.

,Materia Prima,Unidad,Precio por Unidad,Requerimiento por Docena,Costo por docena
0,Harina,Kg,1500.0,0.490,735.0
1,Leche,Litro,2000.0,0.230,460.0
2,Manteca,kg,5000.0,0.200,1000.0
3,Azucar,kg,1000.0,0.050,50.0
4,Huevo,Unidad,150.0,3.000,450.0
5,Levadura,Kg,1500.0,0.015,22.5
6,"Sal, Escencias, Etc.",Kg,1000.0,0.015,15.0
7,Costo Materia Prima Unitario,NaN,NaN,NaN,2732.5
8,Costo Materia Prima Total,NaN,NaN,NaN,136625000.0


##COSTOS INDIRECTOS DE FABRIACIÓN (CIF) TOTAL Y UNITARIOS

In [13]:
# ==========================================
# 3. PROCESAMIENTO: COSTOS INDIRECTOS (CIF)
# ==========================================
# Cargar archivo de CIF
df_Costos_Indirectos = pd.read_excel('COSTOS_INDIRECTOS.xlsx')

In [14]:
# Limpieza de datos
df_Costos_Indirectos.columns = df_Costos_Indirectos.columns.str.strip() #Limpiar espacios en los nombres de las columnas
df_Costos_Indirectos = df_Costos_Indirectos.dropna(how='all', axis=1) #Eliminar columnas que queden completamente vacías.
df_Costos_Indirectos = df_Costos_Indirectos.dropna(subset=['Costos Indirectos']) # Limpiar filas vacías que puedan quedar al final.

In [15]:
df_Costos_Indirectos.head(10)

,Costos Indirectos,Monto
0,Alquiler Planta Productiva,3000000
1,Luz,1000000
2,Depreciación Maquinas,100000
3,Depreciacion Muebles y Utiles,40000


In [16]:
# Cálculos
# Calcular la suma de la columna Monto (Total de Costos Indirectos)
total_monto_indirectos = df_Costos_Indirectos['Monto'].sum()

# Calcular el CIF Unitario usando las docenas del primer input
# Usamos un condicional por si se ingresó 0 en el input y evitar un error de división por cero
if Docenas_Elaboradas > 0:
    cif_unitario = total_monto_indirectos / Docenas_Elaboradas
else:
    cif_unitario = 0

# Agregar filas de totales
# Crear las dos nuevas filas
fila_total_indirectos = {
    'Costos Indirectos': 'Total Costos Indirectos',
    'Monto': total_monto_indirectos
}

fila_cif_unitario = {
    'Costos Indirectos': 'CIF Unitario',
    'Monto': cif_unitario
}

# Agregar ambas filas en orden al final del DataFrame
df_Costos_Indirectos = pd.concat([df_Costos_Indirectos, pd.DataFrame([fila_total_indirectos, fila_cif_unitario])], ignore_index=True)

In [17]:
df_Costos_Indirectos.head(10)

,Costos Indirectos,Monto
0,Alquiler Planta Productiva,3000000.0
1,Luz,1000000.0
2,Depreciación Maquinas,100000.0
3,Depreciacion Muebles y Utiles,40000.0
4,Total Costos Indirectos,4140000.0
5,CIF Unitario,82.8


##MANO DE OBRA DIRECTA (M.O.D)

La empresa analizada emplea actualmente 10 trabajadores (lo cual puede variar mensualmente según la necesidad operativa)  dedicados directa y exclusivamente a la producción, a los cuales se le paga por docena terminada, el costo por docena de MOD por persona es de ($30).

In [18]:
# ==========================================
# 4. PROCESAMIENTO: MANO DE OBRA DIRECTA (MOD)
# ==========================================
# Cargar archivo de MOD
df_Mano_Obra_Directa = pd.read_excel('MANO_OBRA_DIRECTA.xlsx')

In [19]:
# Limpieza de datos
df_Mano_Obra_Directa.columns = df_Mano_Obra_Directa.columns.str.strip() # Limpiar espacios en los nombres de las columnas
df_Mano_Obra_Directa = df_Mano_Obra_Directa.dropna(how='all', axis=1)   # Eliminar columnas que queden completamente vacías.
df_Mano_Obra_Directa = df_Mano_Obra_Directa.dropna(subset=['Cantidad de Operarios']) # Limpiar filas vacías basándose en la columna 'Cantidad de Operarios'

In [20]:
# Cálculos
# Calcular el Costo de Mano de Obra Unitario (Suma de la multiplicación de ambas columnas)
# Esto multiplica cada fila (operarios x su costo por docena) y suma todos los resultados
costo_mo_unitario_total = (df_Mano_Obra_Directa['Cantidad de Operarios'] * df_Mano_Obra_Directa['Costo por Docena Producida']).sum()

# Calcular el Costo de Mano de Obra Total usando el input de docenas elaboradas
costo_mo_total_mes = costo_mo_unitario_total * Docenas_Elaboradas

# Agregar filas de totales
# Crear las dos nuevas filas
fila_mo_unitaria = {
    'Cantidad de Operarios': 'Costo de Mano de Obra Unitario por docena',
    'Costo por Docena Producida': costo_mo_unitario_total
}

fila_mo_total = {
    'Cantidad de Operarios': 'Costo de Mano de Obra Total',
    'Costo por Docena Producida': costo_mo_total_mes
}

# Agregar ambas filas al final del DataFrame de Mano de Obra
df_Mano_Obra_Directa = pd.concat([df_Mano_Obra_Directa, pd.DataFrame([fila_mo_unitaria, fila_mo_total])], ignore_index=True)

In [21]:
df_Mano_Obra_Directa.head()

,Cantidad de Operarios,Costo por Docena Producida
0,10,30.0
1,Costo de Mano de Obra Unitario por docena,300.0
2,Costo de Mano de Obra Total,15000000.0


##Costos Fijos

In [22]:
# ==========================================
# 5. PROCESAMIENTO: COSTOS FIJOS
# ==========================================
# Cargar el archivo Excel de Costos Fijos
df_Costos_Fijos = pd.read_excel('COSTOS_FIJOS.xlsx')

In [23]:
# Limpieza de datos (Protección contra duplicación de totales)
# Limpieza estándar
df_Costos_Fijos.columns = df_Costos_Fijos.columns.str.strip()
df_Costos_Fijos = df_Costos_Fijos.dropna(how='all', axis=1)

# --- ESTA LÍNEA CORRIGE EL ERROR ---
# Elimina cualquier fila que ya se llame 'Costos Fijos Total' para empezar limpio
df_Costos_Fijos = df_Costos_Fijos[df_Costos_Fijos['Costos Fijos'] != 'Costos Fijos Total']

# Limpiar las filas vacías reales
df_Costos_Fijos = df_Costos_Fijos.dropna(subset=['Costos Fijos'])

# Cálculos
#  Calcular la suma total limpia
total_costos_fijos = df_Costos_Fijos['Monto'].sum()

# Agregar Fila de Total
# Crear la nueva fila con el concepto del total
fila_total_fijos = {
    'Costos Fijos': 'Costos Fijos Total',
    'Monto': total_costos_fijos
}

# Agregar la fila al final
df_Costos_Fijos = pd.concat([df_Costos_Fijos, pd.DataFrame([fila_total_fijos])], ignore_index=True)

In [24]:
df_Costos_Fijos.head(10)

,Costos Fijos,Monto
0,Alquiler Planta Productiva,3000000
1,Impuestos Provinciales y Municipales,250000
2,Depreciación Maquinas,100000
3,Depreciacion Muebles y Utiles Productivos,40000
4,Depreciacion Muebles y Utiles Administracíon,30000
5,Publicidad,500000
6,Costos Fijos Total,3920000


##CÁLCULO DE INDICADORES FINANCIEROS Y EQUILIBRIO

##Costo Total de Producción.

In [25]:
# ==========================================
# 6. CÁLCULO DE INDICADORES FINANCIEROS Y EQUILIBRIO
# ==========================================
# Calcular el Costo Total de Producción sumando los totales de las tres áreas
costo_total_produccion = Costo_Materia_Prima_Total + costo_mo_total_mes + total_monto_indirectos

# 2. Imprimir el resultado en la terminal con formato de moneda legible
print(f"El Costo Total de Producción es de ${costo_total_produccion:,.2f}")

El Costo Total de Producción es de $155,765,000.00


###Costo Unitario de Producción.

In [26]:
# Calcular el Costo Unitario de Producción
# Se valida que Docenas_Elaboradas sea mayor a cero para evitar error de división por cero
if Docenas_Elaboradas > 0:
    costo_unitario_produccion = costo_total_produccion / Docenas_Elaboradas
else:
    costo_unitario_produccion = 0

# Imprimir el resultado
print(f"El Costo Unitario de Producción por docena es de ${costo_unitario_produccion:,.2f}")

El Costo Unitario de Producción por docena es de $3,115.30


##Costos Directos/Primos  MP+MOD

In [27]:
# Calcular Costos Directos (Totales y Unitarios)
costos_directos_total = Costo_Materia_Prima_Total + costo_mo_total_mes

if Docenas_Elaboradas > 0:
    costos_directos_unitario = costos_directos_total / Docenas_Elaboradas
else:
    costos_directos_unitario = 0

print(f"Los Costos Directos Totales son de ${costos_directos_total:,.2f} (Unitario: ${costos_directos_unitario:,.2f} por docena)")

Los Costos Directos Totales son de $151,625,000.00 (Unitario: $3,032.50 por docena)


##Costos de Conversión MOD + CIF

In [28]:
# Calcular Costos de Conversión (Totales y Unitarios)
costos_conversion_total = costo_mo_total_mes + total_monto_indirectos

if Docenas_Elaboradas > 0:
    costos_conversion_unitario = costos_conversion_total / Docenas_Elaboradas
else:
    costos_conversion_unitario = 0

print(f"Los Costos de Conversión Totales son de ${costos_conversion_total:,.2f} (Unitario: ${costos_conversion_unitario:,.2f} por docena)")

Los Costos de Conversión Totales son de $19,140,000.00 (Unitario: $382.80 por docena)


##Margen de Contribución - Precio de Equilibrio - Cantidad de Equilibrio.

####Ingresamos el Precio de Venta Comercial.

In [29]:
# Solicitar únicamente el precio de venta estimado
try:
    precio_venta_estimado = float(input("Ingrese el Precio de Venta estimado por docena: "))
except ValueError:
    print("Error: Ingrese un valor numérico válido.")
    precio_venta_estimado = 0

Ingrese el Precio de Venta estimado por docena: 7000


####Calculamos Precio y Cantidad de Equilibrio, Costo variable Unitario y Margen de Contribución.

In [30]:
# 1. El costo variable por docena ya está guardado en esta variable (Materia Prima + Mano de Obra)
costo_variable_unitario = costos_directos_unitario


# 2 . Cantidad de Equilibrio (Docenas necesarias)
margen_contribucion = precio_venta_estimado - costo_variable_unitario

if margen_contribucion > 0:
    cantidad_equilibrio = total_costos_fijos / margen_contribucion
else:
    cantidad_equilibrio = 0

# 3 . Precio de Equilibrio (Precio mínimo por docena para cubrir todo)
if Docenas_Elaboradas > 0:
    precio_equilibrio = costo_variable_unitario + (total_costos_fijos / Docenas_Elaboradas)
else:
    precio_equilibrio = 0

print(f"\n--- REPORTE DE EQUILIBRIO ---")

print(f"Costo Variable Unitario: {costo_variable_unitario:,.2f} ")
print(f"Margen de Contribución: {margen_contribucion:,.2f} ")
print(f"Cantidad de Equilibrio: {cantidad_equilibrio:,.2f} Docenas")
print(f"Precio de Equilibrio: ${precio_equilibrio:,.2f} por Docena")


--- REPORTE DE EQUILIBRIO ---
Costo Variable Unitario: 3,032.50 
Margen de Contribución: 3,967.50 
Cantidad de Equilibrio: 988.03 Docenas
Precio de Equilibrio: $3,110.90 por Docena


In [31]:
# ==========================================
# 7. EXPORTACIÓN CON FORMATO PROFESIONAL (CORREGIDO FINAL V2)
# ==========================================
# Validar que el precio de equilibrio no quede negativo si el margen es a pérdida
precio_equilibrio_limpio = max(0, precio_equilibrio)

# Estructurar la hoja resumen
datos_resumen = {
    'Indicador Financiero': [
        'Total Docenas Elaboradas este Mes',
        'Costo Total de Producción',
        'Costo Unitario de Producción',
        'Costos Directos Totales (Primos)',
        'Costos Directos Unitarios',
        'Costos de Conversión Totales',
        'Costos de Conversión Unitarios',
        'Cantidad de Equilibrio (Punto de Equilibrio)',
        'Precio de Equilibrio (Precio Mínimo)'
    ],
    'Valor': [
        Docenas_Elaboradas, costo_total_produccion, costo_unitario_produccion,
        costos_directos_total, costos_directos_unitario, costos_conversion_total,
        costos_conversion_unitario, cantidad_equilibrio, precio_equilibrio_limpio
    ]
}
df_Resumen_Final = pd.DataFrame(datos_resumen)

nombre_salida_excel = 'Gestion_Costos_y_Automatizacion.xlsx'

# Guardar los DataFrames en pestañas independientes
with pd.ExcelWriter(nombre_salida_excel, engine='openpyxl') as writer:
    df_Materia_Prima.to_excel(writer, sheet_name='Materia Prima', index=False)
    df_Costos_Indirectos.to_excel(writer, sheet_name='Costos Indirectos CIF', index=False)
    df_Mano_Obra_Directa.to_excel(writer, sheet_name='Mano de Obra Directa', index=False)
    df_Costos_Fijos.to_excel(writer, sheet_name='Costos Fijos Estructura', index=False)
    df_Resumen_Final.to_excel(writer, sheet_name='Resumen de Indicadores', index=False)

# Aplicar el diseño con openpyxl
wb = openpyxl.load_workbook(nombre_salida_excel)

fuente_titulo = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
fuente_datos = Font(name='Segoe UI', size=10)
fuente_totales = Font(name='Segoe UI', size=10, bold=True)
relleno_titulo = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
relleno_totales = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')

alineacion_centro = Alignment(horizontal='center', vertical='center')
alineacion_izquierda = Alignment(horizontal='left', vertical='center')
alineacion_derecha = Alignment(horizontal='right', vertical='center')

borde_fino = Border(left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'), top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9'))
borde_total = Border(top=Side(style='thin', color='000000'), bottom=Side(style='double', color='000000'))

for nombre_hoja in wb.sheetnames:
    ws = wb[nombre_hoja]
    ws.row_dimensions.height = 25

    # Formatear encabezados
    for celda in ws[1]: # Acceder directamente a la fila 1 de celdas
        celda.font = fuente_titulo
        celda.fill = relleno_titulo
        celda.alignment = alineacion_centro

    # Formatear filas de datos y totales
    for fila in range(2, ws.max_row + 1):
        ws.row_dimensions[fila].height = 18
        texto_primer_columna = str(ws.cell(row=fila, column=1).value).lower()
        es_fila_total = any(p in texto_primer_columna for p in ['total', 'unitario', 'equilibrio', 'producción', 'costos directos', 'conversión'])

        for col in range(1, ws.max_column + 1):
            celda = ws.cell(row=fila, column=col)
            celda.font = fuente_totales if es_fila_total else fuente_datos
            if es_fila_total:
                celda.fill = relleno_totales
                celda.border = borde_total
            else:
                celda.border = borde_fino

            valor = celda.value
            if isinstance(valor, (int, float)):
                nombre_columna = str(ws.cell(row=1, column=col).value).lower()

                es_indicador_cantidad = any(p in nombre_columna for p in ['cantidad', 'operarios', 'docenas', 'requerimiento'])
                es_fila_docenas_resumen = 'docenas' in texto_primer_columna or 'punto de equilibrio' in texto_primer_columna

                if es_indicador_cantidad or es_fila_docenas_resumen:
                    celda.number_format = '#,##0.00'
                    celda.alignment = alineacion_centro
                else:
                    celda.number_format = '"$"#,##0.00'
                    celda.alignment = alineacion_derecha
            else:
                celda.alignment = alineacion_izquierda

    # CORRECCIÓN EN EL AUTOAJUSTE DE COLUMNAS (Evita el AttributeError de la tupla)
    for col in ws.columns:
        max_len = max(len(str(celda.value or '')) for celda in col)
        # Extraemos el atributo 'column' de la primera celda de la tupla de la columna
        numero_columna = col[0].column
        col_letter = get_column_letter(numero_columna)
        ws.column_dimensions[col_letter].width = max(max_len + 7, 16)

wb.save(nombre_salida_excel)
print(f"\n¡Sección 7 actualizada y corregida! El archivo '{nombre_salida_excel}' se procesó con éxito.")


¡Sección 7 actualizada y corregida! El archivo 'Gestion_Costos_y_Automatizacion.xlsx' se procesó con éxito.


In [32]:
from google.colab import files
files.download('Gestion_Costos_y_Automatizacion.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###                           ***NIETO JORGE OSCAR***